In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd

In [ ]:
df = pd.read_excel('ФакторыРаботы.xlsx')
df = df.drop('Unnamed: 0', axis=1)


In [ ]:
df

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14
0,52,27,49,35,27,19,25,14,9,13,8,6,7,3
1,55,29,49,35,23,20,21,15,9,13,10,6,6,2
2,53,27,46,34,25,22,24,11,9,11,10,6,5,2
3,59,38,35,31,27,23,19,10,13,11,10,6,5,2
4,56,37,35,29,28,25,23,13,12,11,10,9,5,2


In [ ]:
class SVDPCA:
    def __init__(self, n_components=None):
        self.n_components = n_components # количество компонентов (в данном случае их 14)

    @staticmethod # cтатический метод для "выравнивания" векторов
    def svd_flip_vector(U):
        max_abs_cols_U = np.argmax(np.abs(U), axis=0)
        signs_U = np.sign(U[max_abs_cols_U, range(U.shape[1])])

        return U * signs_U

    def fit_transform(self, X):
        n_samples, n_features = X.shape
        X_centered = X - X.mean(axis=0) # "центрируем" данные, вычитая среднее значение по каждому признаку (столбцу)

        if self.n_components is None:
            self.n_components = min(n_samples, n_features) # если количество компонент не указано,
            #мы выбираем минимальное значение между количеством образцов и количеством признаков

        U, S, Vt = np.linalg.svd(X_centered) # на центрированных данных X_centered получаем три матрицы: U, S и Vt
        U_flipped = self.svd_flip_vector(U)

        self.explained_variance = (S[:self.n_components] ** 2) / (n_samples - 1) # объясненная дисперсия, которая пропорциональна квадратичным значениям сингулярных значений S для первых n_components
        self.explained_variance_ratio = self.explained_variance / np.sum(self.explained_variance)

        # X_new = X * V = U * S * Vt * V = U * S
        X_transformed = U_flipped[:, : self.n_components] * S[: self.n_components] # получаем преобразованные данные, умножая U_flipped на соответствующие сингулярные значения S

        return X_transformed

In [ ]:
pca = SVDPCA()
X_transformed = pca.fit_transform(df)

print('transformed data', X_transformed, '', sep='\n')
print('explained_variance', pca.explained_variance)
print('explained_variance_ratio', pca.explained_variance_ratio)
print(sum(pca.explained_variance_ratio))

transformed data
[[-9.42859356e+00 -2.05111245e+00 -1.32537178e+00  2.24898885e+00
   2.56563029e-15]
 [-7.40504613e+00  3.82693746e+00 -1.51106583e+00 -1.53509660e+00
   2.56563029e-15]
 [-5.64458932e+00 -1.53469086e+00  3.37093645e+00 -9.69599292e-01
   2.56563029e-15]
 [ 1.17405568e+01  2.91764657e+00  1.07781239e+00  1.61399693e+00
   2.56563029e-15]
 [ 1.07376723e+01 -3.15878072e+00 -1.61231123e+00 -1.35828989e+00
   2.56563029e-15]]

explained_variance [1.07183188e+02 9.92458646e+00 4.79109248e+00 3.20113318e+00
 8.22807346e-30]
explained_variance_ratio [8.56780079e-01 7.93332251e-02 3.82981014e-02 2.55885946e-02
 6.57719701e-32]
1.0
